<font size=10>**PREPROCESSING**</font> <a class="anchor" id='title'></a> 

**Bachelor's in Data Science - NOVA IMS (25/26)**

**Data**: 
- [*Portal BASE*](https://www.base.gov.pt/Base4/pt/pesquisa/?type=contratos&texto=&adjudicante=&adjudicataria=&tipo=2&tipocontrato=0&cpv=&aqinfo=&desdeprazoexecucao=&ateprazoexecucao=&sel_price=price_11&desdeprecocontrato=&ateprecocontrato=&desdeprecoefectivo=&ateprecoefectivo=&sel_date=date_11&desdedatacontrato=2023-01-01&atedatacontrato=2026-03-31&desdedatapublicacao=&atedatapublicacao=&desdedatafecho=&atedatafecho=&pais=0&distrito=0&concelho=0)

- [*Treated Datasets*](https://dados.gov.pt/pt/datasets/contratos-publicos-portal-base-impic-contratos-de-2012-a-2026/#/resources)

**Group B**
- Beatriz Marques 20231605
- Maria Inês Santos 20231630
- Luís Soeiro 20211536
- Rodrigo Silva 20231602

<font color='#BFD72' size=6>**TABLE OF CONTENTS**</font> <a class="anchor" id='toc'></a>  
- [1. Imports](#1)  
- [2. Data Integration](#2)  
- [3. Data Preprocessing](#3) 

# <font color='#BFD72F' size=6>**1. Imports**</font> <a class="anchor" id="1"></a>

[Back to TOC](#toc)

In [1]:
import warnings
%load_ext autoreload
%autoreload 2

warnings.filterwarnings('ignore')

In [2]:
import sys
import os

# Get the absolute path of the source_code folder
source_code_path = os.path.abspath('../source')

# Add the source_code folder to sys.path
if source_code_path not in sys.path:
    sys.path.append(source_code_path)

In [3]:
import subprocess, sys, importlib
import pandas as pd
import plotly.express as px

try:
    import openpyxl
except ImportError:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "openpyxl"])
    importlib.invalidate_caches()
    import openpyxl
import os
import shutil

# <font color='#BFD72F' size=6>**2. Data Integration**</font> <a class="anchor" id="2"></a>
  
[Back to TOC](#toc)

In [4]:
# MERGE DATASETS
paths = {
    "2023_part01": "../data/contratos2023_part01.csv",
    "2023_part02": "../data/contratos2023_part02.csv",
    "2023_part03": "../data/contratos2023_part03.csv",
    "2024_part01": "../data/contratos2024_part01.csv",
    "2024_part02": "../data/contratos2024_part02.csv",
    "2024_part03": "../data/contratos2024_part03.csv",
    "2025_part01": "../data/contratos2025_part01.csv",
    "2025_part02": "../data/contratos2025_part02.csv",
    "2025_part03": "../data/contratos2025_part03.csv",
    "2026": "../data/contratos2026.csv",
}

datasets = {}

merged_dataset = pd.DataFrame()

for year, path in paths.items():
    print(f"Loading dataset for {year} from {path}...")
    datasets[year] = pd.read_csv(path)
    print(f"Dataset for {year} loaded successfully with shape {datasets[year].shape}.")
    merged_dataset = pd.concat([merged_dataset, datasets[year]], ignore_index=True)

print(f"Merged dataset created with shape {merged_dataset.shape}.")

Loading dataset for 2023_part01 from ../data/contratos2023_part01.csv...
Dataset for 2023_part01 loaded successfully with shape (64562, 35).
Loading dataset for 2023_part02 from ../data/contratos2023_part02.csv...
Dataset for 2023_part02 loaded successfully with shape (64562, 35).
Loading dataset for 2023_part03 from ../data/contratos2023_part03.csv...
Dataset for 2023_part03 loaded successfully with shape (64562, 35).
Loading dataset for 2024_part01 from ../data/contratos2024_part01.csv...
Dataset for 2024_part01 loaded successfully with shape (75440, 35).
Loading dataset for 2024_part02 from ../data/contratos2024_part02.csv...
Dataset for 2024_part02 loaded successfully with shape (75440, 35).
Loading dataset for 2024_part03 from ../data/contratos2024_part03.csv...
Dataset for 2024_part03 loaded successfully with shape (75441, 35).
Loading dataset for 2025_part01 from ../data/contratos2025_part01.csv...
Dataset for 2025_part01 loaded successfully with shape (81131, 35).
Loading datas

# <font color='#BFD72F' size=6>**3. Data Preprocessing**</font> <a class="anchor" id="3"></a>
  
[Back to TOC](#toc)

## <font color='#BFD72F' size=6>3.1 Filtering</font> <a class="anchor" id="3.1"></a>
  
[Back to TOC](#toc)

**Filters Applied**: 
- Procurement Procedure Type: Public
- Contract Start Date: 2023-01-01   
- Contract End Date: 2026-04-25 

In [5]:
subset = merged_dataset[merged_dataset['tipoprocedimento'] == 'Concurso público']

cols_to_keep = [
    'idcontrato', 'tipoContrato', 'CPV', 
    'adjudicante', 'adjudicatarios', 'concorrentes', 
    'dataPublicacao', 'dataCelebracaoContrato', 
    'precoBaseProcedimento', 'precoContratual', 'PrecoTotalEfetivo', 
    'LocalExecucao'
]

subset = subset[cols_to_keep]

In [6]:
subset.shape

(113185, 12)

In [7]:
print("There are {} Public Entities.".format(subset['adjudicante'].nunique()))
print("There are {} Companies.".format(subset['adjudicatarios'].nunique()))

print("So, in total our analysis contains {} Nodes.".format(
    subset['adjudicatarios'].nunique() + 
    subset['adjudicante'].nunique()))

There are 4015 Public Entities.
There are 26110 Companies.
So, in total our analysis contains 30125 Nodes.


In [8]:
print("{} Contracts, characterised by {} columns.".format(subset.shape[0], subset.shape[1]))

113185 Contracts, characterised by 12 columns.


## <font color='#BFD72F' size=6>3.2 Drop Duplicates</font> <a class="anchor" id="3.2"></a>
  
[Back to TOC](#toc)

In [9]:
# Check duplicated rows in the merged dataset
duplicated_rows = subset.duplicated()

print(f"Number of duplicated rows: {duplicated_rows.sum()}")

Number of duplicated rows: 505


In [10]:
subset.shape

(113185, 12)

In [11]:
# TODO: drop duplicated rows
subset = subset.drop_duplicates()

In [12]:
subset.shape

(112680, 12)

## <font color='#BFD72F' size=6>3.3 Data Types</font> <a class="anchor" id="3.3"></a>
  
[Back to TOC](#toc)

In [13]:
# TODO: date columns     
subset["dataPublicacao"] = pd.to_datetime(subset["dataPublicacao"], errors='coerce')
subset["dataCelebracaoContrato"] = pd.to_datetime(subset["dataCelebracaoContrato"], errors='coerce')

## <font color='#BFD72F' size=6>3.4 Text Preprocessing</font> <a class="anchor" id="3.4"></a>
  
[Back to TOC](#toc)

In [14]:
# tipoContrato
subset['tipoContrato'] = subset['tipoContrato'].astype(str).str.replace(r'[\r\n]+', ' | ', regex=True).str.strip()

In [15]:
# CPV
subset['CPV'] = subset['CPV'].astype(str).str.replace('\n', ' | ', regex=True).str.strip()
subset['CPV'] = subset['CPV'].astype(str).str.replace(r'\s*\d{8}-\d\s*-\s*', ' ', regex=True).str.strip()

In [16]:
# concorrentes
subset['concorrentes'] = (
    subset['concorrentes']
    .astype(str)
    # replace line breaks with separator
    .str.replace(r'[\r\n]+', ' | ', regex=True)
    # remove excessive spaces
    .str.replace(r'\s+', ' ', regex=True)
    # remove trailing numbers (like " 102", " 89", etc.)
    .str.replace(r'\s+\d+\s*$', '', regex=True)
    # final trim
    .str.strip()
)

In [17]:
# LocalExecucao
subset['LocalExecucao'] = subset['LocalExecucao'].fillna('')

subset['LocalExecucao'] = (
    subset['LocalExecucao']
    # remove "Portugal, " at the start
    .str.replace(r'^Portugal,\s*', '', regex=True)
    # remove standalone "Portugal"
    .str.replace(r'^Portugal$', '', regex=True)
    # replace line breaks
    .str.replace(r'[\r\n]+', ' | ', regex=True)
    # normalize spaces
    .str.replace(r'\s+', ' ', regex=True)
    .str.strip()
    # remove duplicates inside cell
    .apply(lambda x: ' | '.join(dict.fromkeys(x.split(' | '))) if x else x)
)

subset[['district', 'city']] = (
    subset['LocalExecucao']
    .str.split(' \| ', expand=False)
    .str[0]
    .str.split(', ', expand=True)
)

In [18]:
# tudo lower?
# drop LocalExecucao?

In [19]:
subset.head()

,idcontrato,tipoContrato,CPV,adjudicante,adjudicatarios,concorrentes,dataPublicacao,dataCelebracaoContrato,precoBaseProcedimento,precoContratual,PrecoTotalEfetivo,LocalExecucao,district,city
2,9664960,Aquisição de bens móveis,Computadores portáteis,505387271 - Universidade do Algarve,"502163518 - Empis - Informática e Serviços, Lda",501333401-BASE2 - Informática e Telecomunicaçõ...,2023-01-02,2023-01-02,320620.41,3260.00,0.00,"Faro, Faro",Faro,Faro
15,9667338,Aquisição de serviços,Serviços de viagens,600072525 - Direção-Geral da Administração da ...,"514862645 - Dot Viagens e Turismo, Lda",506019608-Smile Viagens e Turismo Unipessoal L...,2023-01-03,2023-01-03,81895.31,81895.31,0.00,,,None
24,9668634,Aquisição de bens móveis,Gasóleo,506613399 - Município de Góis,500433402 - Alves Bandeira SA,500433402-Alves Bandeira SA,2023-01-03,2023-01-02,374112.00,327326.40,343994.23,"Coimbra, Góis",Coimbra,Góis
27,9671613,Empreitadas de obras públicas,Obras de revisão e recuperação,506901173 - Município de Braga,"513129596 - DAMOS COR, LDA.","513129596-DAMOS COR, LDA. | 500553408-Alexandr...",2023-01-04,2023-01-03,318935.89,318343.79,0.00,"Braga, Braga",Braga,Braga
30,9664045,Aquisição de bens móveis,Produtos de panificação,680047360 - Serviços de Ação Social da Univers...,500209634 - Padaria Trinas lda,500209634-Padaria Trinas lda,2023-01-02,2023-01-01,85424.72,84659.05,84659.05,"Braga, Braga",Braga,Braga


## <font color='#BFD72F' size=6>3.5 Outiers</font> <a class="anchor" id="3.5"></a>
  
[Back to TOC](#toc)

## <font color='#BFD72F' size=6>3.6 Missing Values</font> <a class="anchor" id="3.6"></a>
  
[Back to TOC](#toc)

In [20]:
total = len(subset)
missing_counts = subset.isnull().sum()
missing_pct = (missing_counts / total * 100).round(2)

missing_df = pd.DataFrame({
    'missing_count': missing_counts,
    'missing_pct': missing_pct
}).sort_values('missing_pct', ascending=False)

# show only columns with any missing values
missing_df

,missing_count,missing_pct
concorrentes,28145,24.87
adjudicatarios,510,0.45
adjudicante,7,0.01
idcontrato,0,0.00
tipoprocedimento,0,0.00
CPV,1,0.00
tipoContrato,1,0.00
dataPublicacao,0,0.00
dataCelebracaoContrato,0,0.00
precoBaseProcedimento,0,0.00


## <font color='#BFD72F' size=6>3.7 Export Preprocessed Data</font> <a class="anchor" id="3.7"></a>
  
[Back to TOC](#toc)

In [ ]:
# subset.to_csv("../data/preprocessed_data.csv", index=False)